# 12 - Reuse-factor sweep of the FINAL design (Reviewer 2 comment 5; Reviewer 4 comment 6)
The letter currently quotes a reuse-factor sweep of the **earlier build** (requested reuse factors 1 to 64: core latency 13 -> 135 cycles, initiation interval 1 -> 64, up to 31,000 LUTs and 118 BRAM18 for factors 2-16 with the resource strategy against 17,137 LUTs at reuse factor 1; 14,551 and 10,625 LUTs at 32 and 64; DSP count 0-1; identical bit-accurate predictions for every reuse factor from 1 to 128). The letter marks this as PENDING for the new design. This notebook repeats it for the final 8-bit design (round-to-nearest, wrap-around, hidden weights <8,1>, accumulators <16,8>).

**Part A (any Linux/Colab runtime, no Vivado needed):** bit-accurate predictions for reuse factors 1 ... 128 on the original and the augmented test windows - they must be identical - and the reuse factor that hls4ml actually uses per layer.
**Part B (needs Vitis HLS or Vivado HLS on the PATH, Linux/WSL):** C-synthesis of every reuse factor and a table of latency / interval / LUT / FF / DSP / BRAM. If no HLS tool is found the projects are written and zipped, and you run `vitis_hls -f build_prj.tcl "reset=0 csim=0 synth=1 cosim=0 validation=0 export=0 vsynth=0"` inside each project folder on the machine that has the tool (`vivado_hls` for Vivado HLS), then send back each `*_prj/solution1/syn/report/*_csynth.rpt` (and `.xml`).
**Part C (manual, Vivado):** DSP-off implementation, instructions at the end.

**Important:** set `OUT_FMT` below to the output type of the IP that was really built for the board (the notebook that exported it says `(12, 2)`, the board result behaves like `(12, 5)`); the answer is in `firmware/defines.h` of the built IP (`layer6_t`).

**How to run (Google Colab, the same place the other notebooks ran).** Folder `MyDrive/ppg_afib_project/` must contain the two PERform zips (`mimic_perform_af_csv.zip`, `mimic_perform_non_af_csv.zip`, searched recursively). Run the cells top to bottom with *Runtime -> Run all*. **Do not edit the setup cell and do not retrain or replace `ppg_afib_model_augmented.h5`.** When it finishes, *File -> Download -> Download .ipynb* (with the outputs) and send the executed notebook back, together with the CSV that is written to `ppg_afib_project/augmented_window_results/`.
If a number differs from the 'letter' value printed next to it, do **not** change anything to make it match: just send the notebook as it is.

In [1]:
!pip install -q hls4ml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 33.0 MB/s eta 0:00:00


In [2]:
# ============================================================
# SHARED SETUP — AUGMENTED data (4x Cheng et al. 2020), identical to the paper pipeline
# ============================================================
import os, sys, random, glob, json, time, zipfile, shutil, subprocess, warnings
import numpy as np
import pandas as pd
import scipy.signal, scipy.stats
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
warnings.filterwarnings('ignore')

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
def set_seeds(seed=SEED):
    random.seed(seed); np.random.seed(seed); tf.keras.utils.set_random_seed(seed)
set_seeds()

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

PROJECT_DIR = os.environ.get('PPG_PROJECT_DIR', '/content/drive/MyDrive/ppg_afib_project')
OUT_DIR = f'{PROJECT_DIR}/augmented_window_results'
os.makedirs(OUT_DIR, exist_ok=True)

FS = 125
WINDOW_SEC = 20
WINDOW_SAMPLES = FS * WINDOW_SEC   # 2,500 samples

# ---------- 1. FIND + unzip the two PERform CSV zips (searches Drive recursively, like your earlier notebook) ----------
import re
DATA = f'{PROJECT_DIR}/ppg_afib_project'      # folder where you uploaded the two zips (searched first)
AF_ZIP_MANUAL, NON_ZIP_MANUAL = None, None   # OPTIONAL: hard-code the two paths if auto-search picks the wrong files

def _find_zips(root):
    hits = []
    for dp, dns, fns in os.walk(root):
        dns[:] = [d for d in dns if d not in ('.Trash', '.Trash-0')]
        hits += [os.path.join(dp, f) for f in fns if f.lower().endswith('.zip')]
    return [z for z in hits if 'hls4ml' not in os.path.basename(z).lower()]
_is_non = lambda p: 'non' in os.path.basename(p).lower()
_is_af = lambda p: bool(re.search(r'(^|[^a-z])af([^a-z]|$)', os.path.basename(p).lower())) and not _is_non(p)

if not (os.path.isdir('af_data') and os.path.isdir('non_af_data')):
    if AF_ZIP_MANUAL and NON_ZIP_MANUAL:
        af_zip, non_zip = AF_ZIP_MANUAL, NON_ZIP_MANUAL
    else:
        zips = _find_zips(DATA) or _find_zips(os.path.dirname(PROJECT_DIR))
        print('candidate zips:', *zips, sep='\n  ')
        af_zip = next((z for z in zips if _is_af(z)), None); non_zip = next((z for z in zips if _is_non(z)), None)
    assert af_zip and non_zip, f'AF / non-AF zips not found - set AF_ZIP_MANUAL / NON_ZIP_MANUAL (looked under {DATA})'
    print('AF zip    :', af_zip); print('non-AF zip:', non_zip)
    for zp, d in ((af_zip, 'af_data'), (non_zip, 'non_af_data')):
        with zipfile.ZipFile(zp) as z:
            z.extractall(d)

af_files = sorted(glob.glob('af_data/**/mimic_perform_af_*_data.csv', recursive=True))
non_af_files = sorted(glob.glob('non_af_data/**/mimic_perform_non_af_*_data.csv', recursive=True))
print(f'Found {len(af_files)} AF files | {len(non_af_files)} non-AF files')
assert len(af_files) and len(non_af_files), 'CSV files not found after extraction - check the zip contents'

# ---------- 2. Original (un-augmented) 20 s windows, cached to Drive ----------
def load_windows(file_list, label):
    W, L, S = [], [], []
    for idx, fp in enumerate(file_list):
        ppg = pd.read_csv(fp)['PPG'].values.astype(np.float32)
        ppg = ppg[~np.isnan(ppg)]
        for i in range(len(ppg) // WINDOW_SAMPLES):
            W.append(ppg[i * WINDOW_SAMPLES:(i + 1) * WINDOW_SAMPLES])
            L.append(label); S.append(f'{label}_{idx}')   # same subject-ID scheme as the original notebook
    return W, L, S

CACHE = f'{PROJECT_DIR}/original_windows_cache.npz'
if os.path.exists(CACHE):
    _c = np.load(CACHE, allow_pickle=False)
    W_ALL, L_ALL, S_ALL = _c['W'], _c['L'], _c['S']
else:
    w1, l1, s1 = load_windows(af_files, 1)
    w0, l0, s0 = load_windows(non_af_files, 0)
    W_ALL = np.array(w1 + w0, dtype=np.float32)
    L_ALL = np.array(l1 + l0, dtype=np.int32)
    S_ALL = np.array(s1 + s0)
    np.savez_compressed(CACHE, W=W_ALL, L=L_ALL, S=S_ALL)
print(f'Original windows: {W_ALL.shape} | AF: {(L_ALL==1).sum()} | non-AF: {(L_ALL==0).sum()} | subjects: {len(np.unique(S_ALL))}')

# ---------- 3. Peak detection + the 11 HRV features (identical to original; parameters exposed) ----------
def extract_intervals(ppg, fs=FS, prom_frac=0.15, min_dist_s=0.4, plaus=(0.33, 1.5), plaus_frac=0.8):
    min_distance = int(min_dist_s * fs)
    rng = np.ptp(ppg)
    prom = prom_frac * rng if rng > 0 else 0
    peaks, _ = scipy.signal.find_peaks(ppg, distance=min_distance, prominence=prom)
    if len(peaks) < 2:
        return None
    iv = np.diff(peaks / fs)
    ok = (iv > plaus[0]) & (iv < plaus[1])
    if ok.mean() < plaus_frac:
        return None
    return iv

def compute_features(iv):
    if iv is None or len(iv) < 4:
        return None
    mean_iv, std_iv = float(np.mean(iv)), float(np.std(iv))
    rmssd = float(np.sqrt(np.mean(np.diff(iv) ** 2)))
    mn, mx = float(np.min(iv)), float(np.max(iv))
    cv = std_iv / mean_iv if mean_iv > 0 else 0.0
    d = np.abs(np.diff(iv))
    return [mean_iv, std_iv, rmssd, mn, mx, mx - mn, cv, float(len(iv) + 1),
            float(np.median(iv)), float(scipy.stats.skew(iv)), float(np.mean(d > 0.05))]

FEATURE_NAMES = ['mean_iv', 'std_iv', 'rmssd', 'min_iv', 'max_iv', 'range_iv', 'cv_iv', 'n_beats', 'median_iv', 'skew_iv', 'pnn50']

def build_features(W, L, S, **peak_kw):
    """Returns X, y, subj, kept_index (windows rejected by the quality gate are dropped, as in the original)."""
    X, y, s, kept = [], [], [], []
    for i in range(len(W)):
        f = compute_features(extract_intervals(W[i], **peak_kw))
        if f is not None:
            X.append(f); y.append(L[i]); s.append(S[i]); kept.append(i)
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32), np.array(s), np.array(kept)

# ---------- 3b. Cheng et al. (2020) augmentation + quality gate, IDENTICAL to data_augmentation_pipeline.ipynb ----------
def apply_cheng_data_augmentation(segment, fs=FS):
    """original + Gaussian noise (5% of window SD) + amplitude scaling N(1, 0.15) + 0.2 Hz baseline wander (20% of window SD)"""
    out = [segment]
    out.append(segment + np.random.normal(0, 0.05 * np.std(segment), len(segment)))
    out.append(segment * np.random.normal(1.0, 0.15))
    t = np.linspace(0, len(segment) / fs, len(segment))
    out.append(segment + 0.2 * np.std(segment) * np.sin(2 * np.pi * 0.2 * t))
    return out

def build_features_augmented(W, L, S):
    X, y, s, dropped = [], [], [], {0: 0, 1: 0}
    for i in range(len(W)):                       # AF windows first, then non-AF (same order as the original notebook)
        for v in apply_cheng_data_augmentation(W[i], fs=FS):
            f = compute_features(extract_intervals(v))
            if f is not None:
                X.append(f); y.append(L[i]); s.append(S[i])
            else:
                dropped[int(L[i])] += 1
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32), np.array(s), dropped

np.random.seed(42); random.seed(42)               # same RNG state as CELL 1 of data_augmentation_pipeline.ipynb
X_ALL, y_ALL, subj_ALL, DROPPED = build_features_augmented(W_ALL, L_ALL, S_ALL)
print(f'AUGMENTED feature matrix: {X_ALL.shape} | AF: {(y_ALL==1).sum()} | non-AF: {(y_ALL==0).sum()} | dropped: {DROPPED}')
print('EXPECTED (paper): (7706, 11) | AF 4282 | non-AF 3424 | dropped AF 266, non-AF 404')
if X_ALL.shape[0] != 7706:
    print('WARNING: the dataset differs from the one in the paper. Check the CSV order / that the same two zips are used.')

# ---------- 4. Splits ----------
def holdout_subjects():
    """Same subject-level 80/20 stratified split (random_state=42) as CELL 6 of the original notebook."""
    uniq = np.unique(subj_ALL)
    labs = [y_ALL[subj_ALL == s][0] for s in uniq]
    tr, te = train_test_split(uniq, test_size=0.20, random_state=42, stratify=labs)
    return tr, te

def make_subject_folds(subj_array, label_array, n_folds=5, seed=SEED):
    """Stratified (by class) subject -> fold map. Fixed ONCE so every configuration uses identical test subjects."""
    rng = np.random.RandomState(seed)
    fmap = {}
    for cls in (0, 1):
        subs = np.unique(subj_array[label_array == cls]); rng.shuffle(subs)
        for k, s in enumerate(subs):
            fmap[s] = k % n_folds
    return fmap

FOLD_MAP = make_subject_folds(S_ALL, L_ALL, 5)

# ---------- 5. Model + evaluation helpers ----------
def make_mlp(n_in=11):
    return models.Sequential([layers.Input(shape=(n_in,)),
                              layers.Dense(16, activation='relu'), layers.Dropout(0.2),
                              layers.Dense(8, activation='relu'),
                              layers.Dense(1, activation='sigmoid')])

def fit_mlp(Xtr, ytr, epochs=30, seed=SEED):
    tf.keras.backend.clear_session(); set_seeds(seed)
    m = make_mlp(Xtr.shape[1])
    m.compile(optimizer=Adam(0.001), loss='binary_crossentropy', metrics=['accuracy'])
    m.fit(Xtr, ytr, epochs=epochs, batch_size=32, verbose=0)
    return m

def keras_fp(Xtr, ytr, Xte, epochs=30):
    sc = StandardScaler(); a = sc.fit_transform(Xtr); b = sc.transform(Xte)
    return fit_mlp(a, ytr, epochs).predict(b, verbose=0).flatten()

def sklearn_fp(factory):
    def fp(Xtr, ytr, Xte):
        sc = StandardScaler(); a = sc.fit_transform(Xtr); b = sc.transform(Xte)
        clf = factory(); clf.fit(a, ytr)
        if hasattr(clf, 'predict_proba'):
            return clf.predict_proba(b)[:, 1]
        return 1.0 / (1.0 + np.exp(-clf.decision_function(b)))
    return fp

def loso_predict(X, y, subj, fit_predict, verbose=True):
    """Leave-one-subject-out: returns out-of-subject probabilities aligned with X."""
    p = np.zeros(len(y), dtype=np.float64)
    uniq = np.unique(subj)
    for k, s in enumerate(uniq):
        te = subj == s
        p[te] = fit_predict(X[~te], y[~te], X[te])
        if verbose and (k + 1) % 5 == 0:
            print(f'  LOSO {k+1}/{len(uniq)}', end='\r')
    return p

def fold_predict(X, y, subj, fit_predict, fmap=None):
    """Fixed subject-level 5-fold CV; returns pooled out-of-fold probabilities + fold id per sample."""
    fmap = fmap or FOLD_MAP
    fold = np.array([fmap[s] for s in subj]); p = np.zeros(len(y))
    for k in range(max(fmap.values()) + 1):
        te = fold == k
        p[te] = fit_predict(X[~te], y[~te], X[te])
    return p, fold

# ---------- 6. Subject-clustered statistics (windows from one subject are NOT independent) ----------
def subject_counts(y, p, subj, thr=0.5):
    pred = (p >= thr).astype(int); rows = []
    for s in np.unique(subj):
        m = subj == s; yy, pp = y[m], pred[m]
        assert len(np.unique(yy)) == 1, 'a subject has mixed labels'
        rows.append((s, int(yy[0]), int(((pp == 1) & (yy == 1)).sum()), int(((pp == 1) & (yy == 0)).sum()),
                     int(((pp == 0) & (yy == 0)).sum()), int(((pp == 0) & (yy == 1)).sum())))
    return pd.DataFrame(rows, columns=['subject', 'label', 'tp', 'fp', 'tn', 'fn'])

def _m(tp, fp, tn, fn):
    n = tp + fp + tn + fn
    sens = tp / (tp + fn) if tp + fn else np.nan
    spec = tn / (tn + fp) if tn + fp else np.nan
    prec = tp / (tp + fp) if tp + fp else np.nan
    f1 = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else np.nan
    return {'accuracy': (tp + tn) / n, 'sensitivity': sens, 'specificity': spec, 'precision': prec, 'f1': f1}

def cluster_bootstrap(y, p, subj, n_boot=5000, seed=SEED, thr=0.5, do_auc=True, n_boot_auc=1000):
    """Class-stratified subject bootstrap (resample AF subjects among AF, non-AF among non-AF).
    Returns {metric: (point, lo95, hi95)}; also 'per_subject_acc_mean' (macro average over subjects)."""
    rng = np.random.RandomState(seed)
    C = subject_counts(y, p, subj, thr)
    cnt = C[['tp', 'fp', 'tn', 'fn']].values.astype(float)
    af = np.where(C['label'].values == 1)[0]; nf = np.where(C['label'].values == 0)[0]
    point = _m(*cnt.sum(0))
    persub = ((C.tp + C.tn) / (C.tp + C.fp + C.tn + C.fn)).values
    point['per_subject_acc_mean'] = persub.mean()
    draws = {k: [] for k in point}
    for _ in range(n_boot):
        idx = np.concatenate([rng.choice(af, len(af)), rng.choice(nf, len(nf))])
        r = _m(*cnt[idx].sum(0)); r['per_subject_acc_mean'] = persub[idx].mean()
        for k in r: draws[k].append(r[k])
    out = {k: (point[k], *np.nanpercentile(draws[k], [2.5, 97.5])) for k in point}
    if do_auc:
        subs = C['subject'].values
        by = {s: np.where(subj == s)[0] for s in subs}
        out['auc'] = (roc_auc_score(y, p), np.nan, np.nan)
        a = []
        for _ in range(n_boot_auc):
            idx = np.concatenate([rng.choice(af, len(af)), rng.choice(nf, len(nf))])
            ii = np.concatenate([by[subs[j]] for j in idx])
            if len(np.unique(y[ii])) == 2:
                a.append(roc_auc_score(y[ii], p[ii]))
        out['auc'] = (out['auc'][0], *np.percentile(a, [2.5, 97.5]))
    return out

def paired_bootstrap_diff(y, pA, pB, subj, n_boot=5000, seed=SEED, thr=0.5):
    """Accuracy(A) - Accuracy(B) with a shared subject resample. Returns (diff, lo95, hi95)."""
    rng = np.random.RandomState(seed)
    CA, CB = subject_counts(y, pA, subj, thr), subject_counts(y, pB, subj, thr)
    a = CA[['tp', 'fp', 'tn', 'fn']].values.astype(float); b = CB[['tp', 'fp', 'tn', 'fn']].values.astype(float)
    af = np.where(CA['label'].values == 1)[0]; nf = np.where(CA['label'].values == 0)[0]
    acc = lambda c: (c[0] + c[2]) / c.sum()
    d = []
    for _ in range(n_boot):
        idx = np.concatenate([rng.choice(af, len(af)), rng.choice(nf, len(nf))])
        d.append(acc(a[idx].sum(0)) - acc(b[idx].sum(0)))
    return acc(a.sum(0)) - acc(b.sum(0)), *np.percentile(d, [2.5, 97.5])

def wilson_ci(k, n, z=1.96):
    """NAIVE window-level Wilson interval — anti-conservative here (windows are clustered by subject)."""
    ph = k / n; den = 1 + z ** 2 / n
    c = (ph + z ** 2 / (2 * n)) / den
    h = z * np.sqrt(ph * (1 - ph) / n + z ** 2 / (4 * n ** 2)) / den
    return c - h, c + h

def fmt(t, pct=True):
    f = 100 if pct else 1
    return f'{t[0]*f:.2f} [{t[1]*f:.2f}, {t[2]*f:.2f}]'

print('Setup complete.')

Mounted at /content/drive
candidate zips:
  /content/drive/MyDrive/Classroom/C Programming  (Fall 2025) Section 4/s20250020287_sriramchandar reddy (1).zip
  /content/drive/MyDrive/Classroom/C Programming  (Fall 2025) Section 4/s20250020287_assignment2 sriramchandarreddy.zip
  /content/drive/MyDrive/Classroom/C Programming  (Fall 2025) Section 4/S20250020287_ASSIGNMENT2 SRIRAM.zip
  /content/drive/MyDrive/Classroom/C Programming  (Fall 2025) Section 4/s20250020287_assignment-3 sriram.zip
  /content/drive/MyDrive/Classroom/C Programming  (Fall 2025) Section 4/s20250020287 cp lab assignment-4.zip
  /content/drive/MyDrive/Classroom/C Programming  (Fall 2025) Section 4/s20250020287_assignment-5.zip
  /content/drive/MyDrive/Classroom/C Programming  (Fall 2025) Section 4/s20250020287_assignment-6.zip
  /content/drive/MyDrive/Classroom/C Programming  (Fall 2025) Section 4/s20250020287_sriramchandar reddy.zip
  /content/drive/MyDrive/Classroom/C Programming  (Fall 2025) Section 4/s20250020287_a

In [3]:
def find_h5():
    for root in (PROJECT_DIR, '/content/drive/MyDrive'):
        hits = []
        for dp, dns, fns in os.walk(root):
            dns[:] = [d for d in dns if not d.startswith('.Trash')]
            if 'ppg_afib_model_augmented.h5' in fns: hits.append(os.path.join(dp, 'ppg_afib_model_augmented.h5'))
        if hits: return sorted(hits, key=lambda s: (len(s), s))[0]
    return None

def split_scaled(X, y, subj):
    tr_s, te_s = holdout_subjects()
    tr, te = np.isin(subj, tr_s), np.isin(subj, te_s)
    sc = StandardScaler().fit(X[tr])
    return sc.transform(X[tr]).astype(np.float32), y[tr], sc.transform(X[te]).astype(np.float32), y[te]

# ---------- hls4ml bit-accurate simulation helpers (same recipe as notebooks 07 / 09) ----------
import hls4ml, tempfile
def fx(w, i, rnd='RND', ovf='SAT'): return f'ap_fixed<{w},{i},AP_{rnd},AP_{ovf}>'
U_INT = {16: 6, 12: 5, 6: 4, 4: 3}   # widths that use ONE common fixed<W,I> for every tensor (paper: 16 -> <16,6>, 12 -> <12,5>, 6 -> <6,4>; the 4-bit integer part is an assumption)
def dense_names(model): return [l.name for l in model.layers if isinstance(l, layers.Dense)]

def make_cfg(model, width=8, rnd='RND', ovf='SAT', in_int=5, accum=(16, 8), out=(12, 2), reuse=1, strategy='Latency'):
    """width 8 = the paper's profiled design: input and layer outputs <8,5>, hidden weights/biases <8,1>, output-layer weights/biases <8,2>,
    accumulators <16,8>, output <12,2>. Widths in U_INT use one common fixed<W,I> for every tensor."""
    cfg = hls4ml.utils.config_from_keras_model(model, granularity='name')
    cfg['Model']['ReuseFactor'] = reuse; cfg['Model']['Strategy'] = strategy
    if width in U_INT:
        f = fx(width, U_INT[width], rnd, ovf)
        cfg['Model']['Precision']['default'] = f
        for ln in cfg['LayerName']:
            p = cfg['LayerName'][ln].get('Precision', None)
            if isinstance(p, dict):
                for k in p: p[k] = f
        return cfg
    assert width == 8, 'width must be 8 (profiled) or one of ' + str(list(U_INT))
    names = dense_names(model)
    cfg['Model']['Precision']['default'] = fx(8, 5, rnd, ovf)
    cfg['Model']['Precision']['result'] = fx(*out, rnd, ovf)
    for ln in cfg['LayerName']:
        if 'input' in ln.lower():
            cfg['LayerName'][ln].setdefault('Precision', {})
            if isinstance(cfg['LayerName'][ln]['Precision'], dict): cfg['LayerName'][ln]['Precision']['result'] = fx(8, in_int, rnd, ovf)
    for n in names[:-1]:
        cfg['LayerName'][n]['Precision'].update(weight=fx(8, 1, rnd, ovf), bias=fx(8, 1, rnd, ovf), result=fx(8, 5, rnd, ovf), accum=fx(*accum, rnd, ovf))
        cfg['LayerName'][n]['ReuseFactor'] = reuse
    cfg['LayerName'][names[-1]]['Precision'].update(weight=fx(8, 2, rnd, ovf), bias=fx(8, 2, rnd, ovf), result=fx(*out, rnd, ovf), accum=fx(*accum, rnd, ovf))
    cfg['LayerName'][names[-1]]['ReuseFactor'] = reuse
    return cfg

_SIM_N = [0]
def bit_accurate_predict(model, cfg, X, keep=False):
    """Compile the hls4ml C simulation of this configuration and return the output probabilities for X."""
    _SIM_N[0] += 1
    d = os.path.join(tempfile.gettempdir(), f'hls_sim_{os.getpid()}_{_SIM_N[0]}')
    shutil.rmtree(d, ignore_errors=True)
    h = hls4ml.converters.convert_from_keras_model(model, hls_config=cfg, output_dir=d, backend='Vivado')
    h.compile()
    p = h.predict(np.ascontiguousarray(X, dtype=np.float32)).flatten()
    if not keep: shutil.rmtree(d, ignore_errors=True)
    return p

def acc_of(p, y): return float(np.mean((np.asarray(p) > 0.5) == (np.asarray(y) > 0.5)))
def ptq_acc(model, X, y, **kw): return acc_of(bit_accurate_predict(model, make_cfg(model, **kw), X), y)
print('hls4ml', hls4ml.__version__)

hls4ml 1.3.0


In [4]:
OUT_FMT = (12, 5)          # <- set to the output type that was really built (see layer6_t in firmware/defines.h of the IP used on the board)
RFS_SIM = [1, 2, 4, 8, 16, 32, 64, 128]
RFS_SYN = [1, 2, 4, 8, 16, 32, 64]
def cfg_rf(model, rf):
    return make_cfg(model, 8, rnd='RND', ovf='WRAP', out=OUT_FMT, reuse=rf, strategy='Latency' if rf == 1 else 'Resource')

# original windows: float network trained here; augmented windows: the paper network (the saved .h5)
Xo, yo, so, _ = build_features(W_ALL, L_ALL, S_ALL)
Xtr, ytr, Xte_o, yte_o = split_scaled(Xo, yo, so); mo = fit_mlp(Xtr, ytr)
Xtr2, ytr2, Xte_a, yte_a = split_scaled(X_ALL, y_ALL, subj_ALL)
p = find_h5(); print('paper model:', p); mp = tf.keras.models.load_model(p, compile=False)
print('paper network float accuracy on the 1,394 augmented test windows: %.2f%%  (letter 98.06%%)' % (100 * acc_of(mp.predict(Xte_a, verbose=0).flatten(), yte_a)))

paper model: /content/drive/MyDrive/ppg_afib_project/ppg_afib_model_augmented.h5
paper network float accuracy on the 1,394 augmented test windows: 98.06%  (letter 98.06%)


## Part A - predictions for reuse factors 1 ... 128

In [5]:
res = {}
for name, m, X, y in (('original windows', mo, Xte_o, yte_o), ('augmented windows', mp, Xte_a, yte_a)):
    ref = None; rows = []
    for rf in RFS_SIM:
        pr = bit_accurate_predict(m, cfg_rf(m, rf), X)
        if ref is None: ref = pr
        rows.append(dict(set=name, requested_rf=rf, accuracy=100 * acc_of(pr, y), identical_to_rf1=bool(np.array_equal(pr, ref)), n_windows_different=int(np.sum((pr > .5) != (ref > .5)))))
        print(rows[-1])
    res[name] = pd.DataFrame(rows)
SIM = pd.concat(res.values(), ignore_index=True); display(SIM)

{'set': 'original windows', 'requested_rf': 1, 'accuracy': 97.98850574712644, 'identical_to_rf1': True, 'n_windows_different': 0}
{'set': 'original windows', 'requested_rf': 2, 'accuracy': 97.98850574712644, 'identical_to_rf1': True, 'n_windows_different': 0}
{'set': 'original windows', 'requested_rf': 4, 'accuracy': 97.98850574712644, 'identical_to_rf1': True, 'n_windows_different': 0}
{'set': 'original windows', 'requested_rf': 8, 'accuracy': 97.98850574712644, 'identical_to_rf1': True, 'n_windows_different': 0}
{'set': 'original windows', 'requested_rf': 16, 'accuracy': 97.98850574712644, 'identical_to_rf1': True, 'n_windows_different': 0}
{'set': 'original windows', 'requested_rf': 32, 'accuracy': 97.98850574712644, 'identical_to_rf1': True, 'n_windows_different': 0}
{'set': 'original windows', 'requested_rf': 64, 'accuracy': 97.98850574712644, 'identical_to_rf1': True, 'n_windows_different': 0}
{'set': 'original windows', 'requested_rf': 128, 'accuracy': 97.98850574712644, 'identi

,set,requested_rf,accuracy,identical_to_rf1,n_windows_different
0,original windows,1,97.988506,True,0
1,original windows,2,97.988506,True,0
2,original windows,4,97.988506,True,0
3,original windows,8,97.988506,True,0
4,original windows,16,97.988506,True,0
5,original windows,32,97.988506,True,0
6,original windows,64,97.988506,True,0
7,original windows,128,97.988506,True,0
8,augmented windows,1,97.776184,True,0
9,augmented windows,2,97.776184,True,0


In [6]:
# reuse factor that hls4ml actually uses per dense layer (it moves the requested value to a valid divisor)
act = []
for rf in RFS_SIM:
    d = os.path.join(tempfile.gettempdir(), f'hls_rfprobe_{rf}'); shutil.rmtree(d, ignore_errors=True)
    h = hls4ml.converters.convert_from_keras_model(mp, hls_config=cfg_rf(mp, rf), output_dir=d, backend='Vivado')
    row = {'requested_rf': rf}
    try:
        for l in h.get_layers():
            if l.class_name == 'Dense': row[l.name] = l.get_attr('reuse_factor')
    except Exception as e:
        row['note'] = repr(e)
    act.append(row)
ACT = pd.DataFrame(act); display(ACT)

,requested_rf,dense_18,dense_19,dense_20
0,1,1,1,1
1,2,1,2,2
2,4,1,4,4
3,8,11,8,8
4,16,11,16,8
5,32,22,32,8
6,64,44,64,8
7,128,88,128,8


## Part B - C-synthesis (needs Vitis HLS / Vivado HLS)

In [7]:
BACKEND = 'Vitis' if shutil.which('vitis_hls') else ('Vivado' if shutil.which('vivado_hls') else None)
print('HLS tool found:', BACKEND)
syn = []
proj_dirs = []
for rf in RFS_SYN:
    d = f'hls_final8_rf{rf}'; shutil.rmtree(d, ignore_errors=True); proj_dirs.append(d)
    h = hls4ml.converters.convert_from_keras_model(mp, hls_config=cfg_rf(mp, rf), output_dir=d, backend=BACKEND or 'Vivado')
    h.write()
    if BACKEND is None:
        continue
    try:
        r = h.build(reset=False, csim=False, synth=True, cosim=False, validation=False, export=False, vsynth=False)
        print(rf, r)
        syn.append(dict(requested_rf=rf, **{k: v for k, v in (r.get('CSynthesisReport') or {}).items()}))
    except Exception as e:
        print('build failed for rf', rf, repr(e)); syn.append(dict(requested_rf=rf, error=repr(e)))
if BACKEND is None:
    with zipfile.ZipFile(f'{OUT_DIR}/hls_final8_projects.zip', 'w', zipfile.ZIP_DEFLATED) as z:
        for d in proj_dirs:
            for dp, dns, fns in os.walk(d):
                for f in fns: z.write(os.path.join(dp, f))
    print('No HLS tool here. Wrote', f'{OUT_DIR}/hls_final8_projects.zip', '-> run the synthesis on the machine with Vitis/Vivado HLS (see the top of this notebook).')
else:
    SYN = pd.DataFrame(syn); display(SYN); SYN.to_csv(f'{OUT_DIR}/reuse_factor_synthesis.csv', index=False)

HLS tool found: None
No HLS tool here. Wrote /content/drive/MyDrive/ppg_afib_project/augmented_window_results/hls_final8_projects.zip -> run the synthesis on the machine with Vitis/Vivado HLS (see the top of this notebook).


In [8]:
SIM.to_csv(f'{OUT_DIR}/reuse_factor_predictions.csv', index=False); ACT.to_csv(f'{OUT_DIR}/reuse_factor_actual_per_layer.csv', index=False)
print('saved csvs to', OUT_DIR)

saved csvs to /content/drive/MyDrive/ppg_afib_project/augmented_window_results


## Part C - DSP-off implementation of the final design (manual, in Vivado)
1. Open the Vivado project of the final design (the one that produced `design_1.bit`).
2. *Settings -> Synthesis -> More Options*: `-max_dsp 0`  (Tcl: `set_property -name {STEPS.SYNTH_DESIGN.ARGS.MORE_OPTIONS} -value {-max_dsp 0} -objects [get_runs synth_1]`).
3. Reset runs, then run synthesis, implementation and bitstream at the same 100 MHz clock.
4. Open the implemented design and export `report_utilization`, `report_timing_summary`, `report_power` (the same three files as in the package).
5. Send back the three reports; the letter currently quotes this experiment for the earlier 50 MHz build (0 DSP, 9,329 LUTs, +181 LUTs, setup slack +7.590 ns).